In [1]:
# INITIAL SETUP

# Import necessary libraries
import pandas as pd
import numpy as np
import re
import ast
from pathlib import Path



# PATHS, CONSTANTS & VARIABLES

# Set path to datasets
DATA_DIR = Path("~/code/StellaRodriguesLallement/OSE_Project/Dataset_visualization/src/ose_core/data_ingestion/extracted_datasets").expanduser()


In [2]:
# DATA LOADING

df_signals   = pd.read_csv(DATA_DIR / '08_signals.csv', dtype={'siren': str})

In [3]:
df_signals.shape

(2133, 12)

In [5]:
df = df_signals.copy()

# Convert the 'type' JSON-like string into a dictionary
df['type'] = df['type'].apply(ast.literal_eval)

# Extract fields
df['signal_code'] = df['type'].apply(lambda x: x.get('code'))
df['signal_label'] = df['type'].apply(lambda x: x.get('label'))

df.columns

Index(['company_name', 'siren', 'siret', 'continent', 'country', 'departement',
       'publishedAt', 'isMain', 'type', 'createdAt', 'companies_count',
       'sirets_count', 'signal_code', 'signal_label'],
      dtype='object')

In [6]:
df.head(3)

,company_name,siren,siret,continent,country,departement,publishedAt,isMain,type,createdAt,companies_count,sirets_count,signal_code,signal_label
0,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1575153000013,"[{'id': 6, 'label': 'Europe'}]","[{'id': 72, 'label': 'France'}]","[{'parent': 'Bourgogne-Franche-Comté', 'id': 2...",2021-09-30T00:00:00+02:00,True,"{'code': 'K1', 'id': 32, 'label': 'Investissem...",2020-09-07T15:14:38+02:00,1,1,K1,Investissements
1,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1575153000013,"[{'id': 6, 'label': 'Europe'}]","[{'id': 72, 'label': 'France'}]","[{'parent': 'Bourgogne-Franche-Comté', 'id': 2...",2020-09-08T00:00:00+02:00,True,"{'code': 'L', 'id': 12, 'label': 'Levée de fon...",2020-09-07T15:14:12+02:00,1,1,L,"Levée de fonds, financements & modifs. capital"
2,PAIN D'EPICES MULOT ET PETITJEAN,015751530,1575153000013,NaN,NaN,NaN,2016-09-21T00:00:00+02:00,True,"{'code': 'F', 'id': 6, 'label': 'Développement...",2016-09-20T10:45:13+02:00,1,1,F,Développement de l'activité


In [ ]:

def prepare_signals(df_signals):
    """
    Clean the raw signals table:
    - convert string dicts to Python dict
    - extract signal_code & signal_label
    - normalise timestamps
    """
    df = df_signals.copy()

    # Convert the 'type' JSON-like string into a dictionary
    df['type'] = df['type'].apply(ast.literal_eval)

    # Extract fields
    df['signal_code'] = df['type'].apply(lambda x: x.get('code'))
    df['signal_label'] = df['type'].apply(lambda x: x.get('label'))

    # Parse date
    df['publishedAt'] = pd.to_datetime(df['publishedAt'], utc=True)

    return df[['siren', 'signal_code', 'signal_label', 'publishedAt']]


def filter_recent_signals(df_signals, months=12):
    cutoff = pd.Timestamp.now(tz='UTC') - pd.DateOffset(months=months)
    return df_signals[df_signals['publishedAt'] >= cutoff].copy()

positive_codes = ['B', 'K1', 'F', 'E', 'S']   # Construction, Investissements, Croissance, Création, Lancement produits
negative_codes = ['M', 'O', 'I']              # Licenciement, RJ/LJ, Fermeture

def classify_signal(code):
    if code in positive_codes:
        return 'positive'
    elif code in negative_codes:
        return 'negative'
    else:
        return 'neutral'

def add_valence(df_signals):
    df = df_signals.copy()
    df['signal_valence'] = df['signal_code'].apply(classify_signal)
    return df


def build_signal_features(df_signals):
    """
    Returns:
    - per-siren counts of positive/negative/neutral signals
    - per-siren counts of each signal_code
    """

    # Valence summary
    valence = (
        df_signals.groupby(['siren', 'signal_valence'])
        .size()
        .unstack(fill_value=0)
        .rename(columns={
            'positive': 'n_positive_signals',
            'negative': 'n_negative_signals',
            'neutral':  'n_neutral_signals'
        })
    )

    # Code summary
    code_counts = (
        df_signals.groupby(['siren', 'signal_code'])
        .size()
        .unstack(fill_value=0)
        .add_prefix('n_code_')
    )

    # Merge both
    return valence.join(code_counts, how='outer').fillna(0)